# Transformación de propiedades usadas

Preparación reproducible de `PropUsadas_Integrado.csv` → `PropUsadas_OLS.csv`.
Se reutilizan las conversiones binarias, exclusiones, tendencia temporal y criterios de
rezago de `Ventas/Propiedades_Usadas/Modelo_OLS.ipynb`, sin modificar ese notebook.
La regla de superficies proviene de `1_EDA_PropUsadas.ipynb` (limpieza 2.2).
No se repiten las integraciones originales ni se ejecutan experimentos.

El objetivo conserva su escala en RUB. Solo se aplican reglas determinísticas;
el preprocesamiento aprendido queda pendiente para cada fold de Train.
Los archivos generados se ubican en `Data/Propiedades_Usadas`.


## 1. Carga e inspección inicial
La entrada permanece en `Ventas/Propiedades_Usadas`. Las rutas funcionan desde la raíz
o desde cualquier subcarpeta del proyecto. Un identificador repetido o una fecha inválida
detiene el proceso para evitar decisiones silenciosas sobre la identidad o la cronología.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

raiz = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "Data").is_dir() and (p / "Ventas").is_dir()
             and (p / "requirements.txt").is_file()), None)
assert raiz is not None, "Ejecutar dentro del proyecto Moscow_EDA."
entrada = raiz / "Ventas" / "Propiedades_Usadas" / "PropUsadas_Integrado.csv"
historico_path = raiz / "Data" / "district_prices_monthly.csv"
salida = raiz / "Data" / "Propiedades_Usadas" / "PropUsadas_OLS.csv"
original = pd.read_csv(entrada)
requeridas = {
    "id", "price_rub", "date_posted_dt", "district", "okrug", "year_month",
    "lat", "lon", "total_area", "living_area", "kitchen_area", "rooms", "floor",
    "total_floors", "building_year", "building_type", "ceiling_height",
    "has_balcony", "renovation", "metro_line", "metro_distance_min",
    "metro_distance_type", "to_center_km", "mortgage_rate_at_listing", "seller_type",
    "metro_lat", "metro_lon", "metro_year_opened", "metro_to_center_km",
}
assert requeridas <= set(original), f"Faltan: {requeridas - set(original)}"
assert original.columns.is_unique and not original.empty
assert original["id"].notna().all() and original["id"].is_unique
print(f"Entrada: {len(original):,} filas × {original.shape[1]} columnas")
print("Filas duplicadas:", original.duplicated().sum())
inspeccion = pd.DataFrame({"tipo": original.dtypes.astype(str),
                           "nulos": original.isna().sum()})
display(inspeccion)
print("Numéricas:", original.select_dtypes(include="number").columns.tolist())
print("Categóricas/binarias:", [c for c in original.select_dtypes(exclude="number")
                               if c not in ["id", "date_posted", "date_posted_dt", "year_month"]])
print("Temporales: date_posted_dt, year_month; años: building_year, metro_year_opened")


Entrada: 50,000 filas × 40 columnas
Filas duplicadas: 0
Numéricas: ['lat', 'lon', 'total_area', 'living_area', 'kitchen_area', 'rooms', 'floor', 'total_floors', 'building_year', 'ceiling_height', 'metro_distance_min', 'to_center_km', 'price_rub', 'price_per_sqm', 'mortgage_rate_at_listing', 'secondary_price_per_sqm', 'newbuild_price_per_sqm', 'secondary_mom_change_pct', 'newbuild_mom_change_pct', 'n_listings_secondary', 'n_listings_newbuild', 'cbr_key_rate_pct', 'avg_mortgage_rate_pct', 'metro_lat', 'metro_lon', 'metro_year_opened', 'metro_to_center_km']
Categóricas/binarias: ['district', 'okrug', 'building_type', 'has_balcony', 'renovation', 'metro_station', 'metro_line', 'metro_distance_type', 'seller_type']
Temporales: date_posted_dt, year_month; años: building_year, metro_year_opened


,tipo,nulos
id,str,0
date_posted,str,0
district,str,0
okrug,str,0
lat,float64,0
lon,float64,0
total_area,float64,0
living_area,float64,0
kitchen_area,float64,0
rooms,int64,0


## 2. Limpieza y transformaciones determinísticas
Se normalizan espacios, Unicode y etiquetas binarias con mapas fijos. Se eliminan únicamente
las superficies inconsistentes con tres medidas finitas, como en el EDA. Otros valores
físicamente imposibles se marcan como faltantes; no se imputan ni se usan percentiles.
Un año de construcción posterior al anuncio se marca mediante `building_year_inconsistent`
y su valor se deja nulo. Precio y fecha deben ser válidos; no se reparan por inferencia.


In [2]:
df = original.copy()
df["date_posted_dt"] = pd.to_datetime(df["date_posted_dt"], format="%Y-%m-%d", errors="raise")
assert df["date_posted_dt"].notna().all()
if "date_posted" in df:
    fecha_original = pd.to_datetime(df["date_posted"], format="%Y-%m-%d", errors="raise")
    assert fecha_original.equals(df["date_posted_dt"]), "Fechas originales discordantes."
assert df["year_month"].equals(df["date_posted_dt"].dt.strftime("%Y-%m"))

categoricas_texto = ["district", "okrug", "building_type", "renovation", "metro_line"]
for columna in categoricas_texto:
    df[columna] = df[columna].astype("string").str.normalize("NFKC").str.strip().replace("", pd.NA)
assert df[["district", "okrug"]].notna().all().all()
assert df.groupby("district")["okrug"].nunique().le(1).all()

numericas_base = [
    "lat", "lon", "total_area", "living_area", "kitchen_area", "rooms", "floor",
    "total_floors", "building_year", "ceiling_height", "metro_distance_min",
    "to_center_km", "mortgage_rate_at_listing", "metro_lat", "metro_lon",
    "metro_year_opened", "metro_to_center_km",
]
for columna in [*numericas_base, "price_rub"]:
    df[columna] = pd.to_numeric(df[columna], errors="raise")
assert df["price_rub"].notna().all() and np.isfinite(df["price_rub"]).all()
assert df["price_rub"].gt(0).all()
df[numericas_base] = df[numericas_base].replace([np.inf, -np.inf], np.nan)

mapas_binarios = {
    "has_balcony": {"false": 0, "true": 1, "0": 0, "1": 1},
    "metro_distance_type": {"walk": 0, "transport": 1},
    "seller_type": {"owner": 0, "agency": 1},
}
for columna, mapa in mapas_binarios.items():
    etiquetas = df[columna].astype("string").str.strip().str.lower().replace("", pd.NA)
    assert etiquetas.dropna().isin(mapa).all(), f"Etiqueta desconocida: {columna}"
    df[columna] = etiquetas.map(mapa).astype("Int64")

areas = ["living_area", "kitchen_area", "total_area"]
inconsistentes = np.isfinite(df[areas]).all(axis=1) & (
    df["living_area"] + df["kitchen_area"] > df["total_area"])
ids_eliminados = df.loc[inconsistentes, "id"].copy()
df = df.loc[~inconsistentes].copy()
reparaciones = {}

def marcar_faltante(columna, mascara):
    reparaciones[columna] = reparaciones.get(columna, 0) + int(mascara.sum())
    df.loc[mascara, columna] = np.nan

for columna in ["total_area", "living_area", "kitchen_area", "ceiling_height"]:
    marcar_faltante(columna, df[columna].le(0))
for columna in ["rooms", "floor", "total_floors", "metro_distance_min",
                "to_center_km", "metro_to_center_km", "mortgage_rate_at_listing"]:
    marcar_faltante(columna, df[columna].lt(0))
marcar_faltante("floor", df["floor"].gt(df["total_floors"]))
for columna, limite in [("lat", 90), ("lon", 180), ("metro_lat", 90), ("metro_lon", 180)]:
    marcar_faltante(columna, df[columna].notna() & ~df[columna].between(-limite, limite))
df["building_year_inconsistent"] = df["building_year"].gt(df["date_posted_dt"].dt.year).astype("int64")
marcar_faltante("building_year", df["building_year_inconsistent"].eq(1))
for columna in ["building_year", "metro_year_opened"]:
    marcar_faltante(columna, df[columna].le(0) | df[columna].gt(df["date_posted_dt"].dt.year))
enteras = ["rooms", "floor", "total_floors", "building_year", "metro_distance_min", "metro_year_opened"]
for columna in enteras:
    assert df[columna].dropna().mod(1).eq(0).all(), f"Valor no entero: {columna}"
    df[columna] = df[columna].astype("Int64")
print("Eliminados por superficies:", len(ids_eliminados))
print("Valores físicamente inconsistentes → nulos:", {c: n for c, n in reparaciones.items() if n})
print("Observaciones conservadas:", len(df))


Eliminados por superficies: 247
Valores físicamente inconsistentes → nulos: {'building_year': 1266}
Observaciones conservadas: 49753


## 3. Características temporales
Se conserva `date_posted_dt` y se deriva `year_month` como clave de auditoría y folds.
`months_since_start` usa el origen fijo enero de 2020 del código anterior;
`month_of_year` representa estacionalidad. No se crean `listing_month` (duplicaría
`month_of_year`) ni `listing_year` (recuperable desde la fecha y la tendencia).


In [3]:
df["year_month"] = df["date_posted_dt"].dt.strftime("%Y-%m")
df["month_of_year"] = df["date_posted_dt"].dt.month.astype("int64")
df["months_since_start"] = ((df["date_posted_dt"].dt.year - 2020) * 12
                             + df["month_of_year"] - 1).astype("int64")
assert df["month_of_year"].between(1, 12).all()


## 4. Indicadores del mes calendario anterior
El integrado carece de `n_listings_rental` y de algunos pares distrito/mes sin anuncios.
Se consulta `Data/district_prices_monthly.csv` únicamente como histórico de indicadores,
sin repetir los JOIN originales de anuncios y estaciones. La clave de búsqueda es el mes
de la propiedad **menos uno**, junto con distrito y okrug. Así un hueco nunca toma
el último mes observado ni se convierte en cero. La tasa se verifica como serie mensual general.

Se excluyen todos los agregados contemporáneos. Se presupone que el mes cerrado anterior
estaba disponible al publicar: la fuente no registra fechas de publicación ni revisiones;
el desfase acredita precedencia del período, pero no disponibilidad efectiva de cada versión.
Los LAG de Validation/Test pueden incorporar meses previos de esos períodos, bajo evaluación
secuencial al publicar cada anuncio, sin utilizar sus precios objetivo.


In [4]:
claves = ["district", "okrug", "year_month"]
origen_lag = ["n_listings_secondary", "n_listings_newbuild", "n_listings_rental", "cbr_key_rate_pct"]
columnas_lag = [f"{c}_lag1" for c in origen_lag]
historico = pd.read_csv(historico_path, usecols=claves + origen_lag)
for columna in ["district", "okrug"]:
    historico[columna] = historico[columna].astype("string").str.normalize("NFKC").str.strip()
assert historico[claves].notna().all().all()
assert not historico.duplicated(claves).any()
historico["_periodo"] = pd.PeriodIndex(historico["year_month"], freq="M")
historico = historico.sort_values(["district", "okrug", "_periodo"], kind="stable")
for columna in origen_lag:
    historico[columna] = pd.to_numeric(historico[columna], errors="raise")
    assert np.isfinite(historico[columna].dropna()).all()
    assert historico[columna].dropna().ge(0).all()
for columna in origen_lag[:3]:
    assert historico[columna].dropna().mod(1).eq(0).all()
assert historico.groupby("year_month")["cbr_key_rate_pct"].nunique(dropna=False).le(1).all()
fuente = historico.set_index(claves)[origen_lag]

# Contrastar lo ya integrado, sin volver a integrar los anuncios.
claves_actuales = pd.MultiIndex.from_frame(df[claves])
actuales = fuente.reindex(claves_actuales)
assert claves_actuales.isin(fuente.index).all()
for columna in origen_lag:
    if columna in df:
        np.testing.assert_allclose(df[columna].to_numpy(dtype=float),
                                   actuales[columna].to_numpy(dtype=float), equal_nan=True)
claves_integradas = pd.MultiIndex.from_frame(original[claves].drop_duplicates())
print("Pares distrito/mes históricos sin anuncios integrados:",
      len(fuente.index.difference(claves_integradas)))

mes_previo = (pd.PeriodIndex(df["year_month"], freq="M") - 1).astype(str)
claves_previas = pd.MultiIndex.from_arrays([df["district"], df["okrug"], mes_previo], names=claves)
valores_previos = fuente.reindex(claves_previas)
for columna, lag in zip(origen_lag, columnas_lag):
    df[lag] = valores_previos[columna].to_numpy(dtype=float)
df["lag1_unavailable"] = df[columnas_lag].isna().any(axis=1).astype("int64")
print("Anuncios con algún LAG no disponible:", int(df["lag1_unavailable"].sum()))
display(df.groupby("year_month")["lag1_unavailable"].agg(["size", "sum"]).query("sum > 0"))


Pares distrito/mes históricos sin anuncios integrados: 50
Anuncios con algún LAG no disponible: 672


,size,sum
year_month,,
2020-01,672,672


## 5. Partición cronológica 70/10/20
Se buscan dos cortes entre meses completos que minimicen la desviación cuadrática de
los tamaños 70/10/20. Solo se cuentan anuncios; no se utilizan precios ni distribuciones
de predictoras. Los porcentajes efectivos pueden variar para preservar meses completos.


In [5]:
df = df.sort_values(["date_posted_dt", "id"], kind="stable").reset_index(drop=True)
conteos_mes = df.groupby("year_month", sort=True).size()
assert len(conteos_mes) >= 3, "Se requieren al menos tres meses."
acumulados = conteos_mes.cumsum().to_numpy()
n = len(df)
candidatos = []
for i in range(len(conteos_mes) - 2):
    for j in range(i + 1, len(conteos_mes) - 1):
        proporciones = np.array([acumulados[i], acumulados[j] - acumulados[i], n - acumulados[j]]) / n
        error = np.square(proporciones - [0.7, 0.1, 0.2]).sum()
        candidatos.append((error, i, j))
_, corte_train, corte_validation = min(candidatos)
ultimo_mes_train = conteos_mes.index[corte_train]
ultimo_mes_validation = conteos_mes.index[corte_validation]
df["dataset_split"] = np.select(
    [df["year_month"].le(ultimo_mes_train), df["year_month"].le(ultimo_mes_validation)],
    ["train", "validation"], default="test")
orden_splits = ["train", "validation", "test"]
resumen_split = df.groupby("dataset_split").agg(
    observaciones=("id", "size"), fecha_min=("date_posted_dt", "min"), fecha_max=("date_posted_dt", "max")
).reindex(orden_splits)
resumen_split["porcentaje"] = resumen_split["observaciones"] / len(df) * 100
assert resumen_split["observaciones"].gt(0).all()
assert resumen_split.loc["train", "fecha_max"] < resumen_split.loc["validation", "fecha_min"]
assert resumen_split.loc["validation", "fecha_max"] < resumen_split.loc["test", "fecha_min"]
assert df.groupby("date_posted_dt")["dataset_split"].nunique().eq(1).all()
assert df.groupby("year_month")["dataset_split"].nunique().eq(1).all()
display(resumen_split)
print("Particiones sin superposición temporal; fechas y meses completos en un único conjunto.")


Particiones sin superposición temporal; fechas y meses completos en un único conjunto.


,observaciones,fecha_min,fecha_max,porcentaje
dataset_split,,,,
train,34853,2020-01-01,2024-05-31,70.052057
validation,5258,2024-06-01,2025-01-31,10.568207
test,9642,2025-02-01,2026-04-29,19.379736


## 6. Características y preprocesamiento pendiente
Se conservan `id` y `okrug` solo para trazabilidad; deben excluirse de X junto con
`price_rub`, `date_posted_dt`, `year_month` y `dataset_split`. `metro_station` se excluye
por redundancia con las características de estación, siguiendo el notebook anterior.
Se eliminan `price_per_sqm`, la fecha duplicada y los diez agregados del mes actual.

One-Hot pendiente: `district`, `building_type`, `renovation`, `metro_line`, `month_of_year`.
El pipeline experimental deberá ajustar imputadores, `OneHotEncoder(drop="first",
handle_unknown="ignore")` y, si se decide usarlo, `StandardScaler`, exclusivamente en
la porción de entrenamiento de **cada fold temporal**. `month_of_year` se trata como
categórica aunque el CSV lo almacene como entero. No se ajusta ningún estimador aquí.
Los folds deben cortar por fecha o mes completo, sin dividir empates de fecha; Validation
confirma finalistas y Test se reserva para la evaluación definitiva.


In [6]:
agregados_actuales = [
    "secondary_price_per_sqm", "newbuild_price_per_sqm", "rental_price_per_sqm_monthly",
    "secondary_mom_change_pct", "newbuild_mom_change_pct", "n_listings_secondary",
    "n_listings_newbuild", "n_listings_rental", "cbr_key_rate_pct", "avg_mortgage_rate_pct",
]
columnas_ohe = ["district", "building_type", "renovation", "metro_line", "month_of_year"]
columnas_numericas = numericas_base + list(mapas_binarios) + [
    "building_year_inconsistent", "months_since_start", *columnas_lag, "lag1_unavailable"]
predictoras = columnas_numericas + columnas_ohe
auxiliares = ["id", "okrug", "date_posted_dt", "year_month", "dataset_split"]
columnas_exportar = ["id", "price_rub", *predictoras, *auxiliares[1:]]
assert len(columnas_exportar) == len(set(columnas_exportar))
columnas_retiradas = sorted(set(df) - set(columnas_exportar))
df_final = df[columnas_exportar].copy()
display(pd.DataFrame([
    {"grupo": "Predictoras numéricas / binarias", "columnas": ", ".join(columnas_numericas)},
    {"grupo": "One-Hot pendiente", "columnas": ", ".join(columnas_ohe)},
    {"grupo": "Auxiliares: excluir de X", "columnas": ", ".join(auxiliares)},
    {"grupo": "Retiradas del CSV", "columnas": ", ".join(columnas_retiradas)},
]))


,grupo,columnas
0,Predictoras numéricas / binarias,"lat, lon, total_area, living_area, kitchen_are..."
1,One-Hot pendiente,"district, building_type, renovation, metro_lin..."
2,Auxiliares: excluir de X,"id, okrug, date_posted_dt, year_month, dataset..."
3,Retiradas del CSV,"avg_mortgage_rate_pct, cbr_key_rate_pct, date_..."


## 7. Validación de integridad
Además del balance de filas y precios, se comprueba el LAG por una vía independiente:
el histórico se traslada exactamente un mes calendario y se cruza con las claves actuales.
Esta unión es solo una comprobación y no modifica los datos exportados.


In [7]:
assert len(df_final) == len(original) - len(ids_eliminados)
assert df_final["id"].is_unique and not df_final.duplicated().any()
assert set(df_final["id"]) == set(original["id"]) - set(ids_eliminados)
precio_fuente = original.set_index("id")["price_rub"].reindex(df_final["id"])
np.testing.assert_array_equal(df_final["price_rub"].to_numpy(), precio_fuente.to_numpy())
assert df_final.columns.is_unique and set(predictoras) <= set(df_final)
assert not set(["price_per_sqm", *agregados_actuales]).intersection(df_final)
assert not set(auxiliares + ["price_rub"]).intersection(predictoras)
assert all(pd.api.types.is_numeric_dtype(df_final[c]) for c in columnas_numericas + ["price_rub"])
for columna in columnas_numericas + ["price_rub"]:
    assert np.isfinite(df_final[columna].dropna().to_numpy(dtype=float)).all()
for columna in [*mapas_binarios, "building_year_inconsistent", "lag1_unavailable"]:
    assert df_final[columna].dropna().isin([0, 1]).all()
assert df_final["date_posted_dt"].is_monotonic_increasing
assert df_final["dataset_split"].isin(orden_splits).all()
assert df_final.groupby("date_posted_dt")["dataset_split"].nunique().eq(1).all()
assert df_final.groupby("year_month")["dataset_split"].nunique().eq(1).all()
assert df_final["year_month"].equals(df_final["date_posted_dt"].dt.strftime("%Y-%m"))
assert df_final["month_of_year"].equals(df_final["date_posted_dt"].dt.month.astype("int64"))
assert df_final["months_since_start"].eq(
    (df_final["date_posted_dt"].dt.year - 2020) * 12 + df_final["date_posted_dt"].dt.month - 1).all()
superficies_presentes = df_final[areas].notna().all(axis=1)
assert (df_final.loc[superficies_presentes, "living_area"]
        + df_final.loc[superficies_presentes, "kitchen_area"]
        <= df_final.loc[superficies_presentes, "total_area"]).all()
assert df_final["building_year"].dropna().le(
    df_final.loc[df_final["building_year"].notna(), "date_posted_dt"].dt.year).all()

control = historico[claves + origen_lag].copy()
control["year_month"] = (pd.PeriodIndex(control["year_month"], freq="M") + 1).astype(str)
control = control.rename(columns=dict(zip(origen_lag, columnas_lag)))
comprobacion = df_final[claves].merge(control, on=claves, how="left", sort=False, validate="many_to_one")
assert len(comprobacion) == len(df_final)
for columna in columnas_lag:
    np.testing.assert_allclose(df_final[columna], comprobacion[columna], equal_nan=True)
assert df_final["lag1_unavailable"].eq(df_final[columnas_lag].isna().any(axis=1).astype(int)).all()
assert (pd.PeriodIndex(df_final["year_month"], freq="M") - 1
        < pd.PeriodIndex(df_final["year_month"], freq="M")).all()
print("Integridad, tipos, precios originales, LAG y separación temporal: OK.")


Integridad, tipos, precios originales, LAG y separación temporal: OK.


## 8. Exportación y comprobación del CSV
Un único CSV, sin índice artificial. La recarga verifica valores, nulos, identidad,
tipos reconstruidos y límites temporales; el formato CSV no conserva metadatos de pandas.


In [8]:
salida.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(salida, index=False, encoding="utf-8", date_format="%Y-%m-%d")
tipos_recarga = {c: "string" for c in ["id", "okrug", "year_month", "dataset_split", *columnas_ohe[:-1]]}
tipos_recarga.update({c: str(df_final[c].dtype) for c in columnas_numericas + ["month_of_year", "price_rub"]})
recargado = pd.read_csv(salida, parse_dates=["date_posted_dt"], dtype=tipos_recarga)
assert recargado.shape == df_final.shape
assert recargado.columns.tolist() == columnas_exportar
assert not any(c.startswith("Unnamed:") for c in recargado)
assert recargado["id"].is_unique
pd.testing.assert_frame_equal(df_final, recargado, check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)
for anterior, siguiente in zip(orden_splits, orden_splits[1:]):
    assert (recargado.loc[recargado["dataset_split"].eq(anterior), "date_posted_dt"].max()
            < recargado.loc[recargado["dataset_split"].eq(siguiente), "date_posted_dt"].min())
print("CSV escrito y recargado correctamente:", salida.relative_to(raiz))


CSV escrito y recargado correctamente: Data\Propiedades_Usadas\PropUsadas_OLS.csv


## 9. Resumen final
El modelado podrá leer este CSV con `parse_dates=["date_posted_dt"]`, seleccionar
`dataset_split` y construir X con `predictoras`, excluyendo las auxiliares documentadas.
No debe volver a limpiar, integrar, rezagar ni repartir los anuncios. Persisten los nulos
que requieren imputación aprendida y las categorías originales para el pipeline de cada fold.
La disponibilidad histórica efectiva de las versiones mensuales sigue siendo un supuesto
de la fuente, documentado en la sección 4.


In [9]:
print(f"Resultado: {len(df_final):,} anuncios, {df_final.shape[1]} columnas, {len(predictoras)} predictoras candidatas.")
print(f"Entrada: {len(original):,}; eliminados por superficies: {len(ids_eliminados):,}.")
print("Objetivo: price_rub en RUB, sin transformación.")
print("Aplicado: tipos, mapas binarios, reglas físicas, tendencia, estacionalidad, LAG y partición mensual.")
print("Pendiente por fold: imputación, One-Hot y eventual escalado / tratamiento estadístico.")
display(resumen_split)
faltantes_finales = df_final.isna().sum()
display(faltantes_finales[faltantes_finales.gt(0)].rename("nulos_pendientes").to_frame())


Resultado: 49,753 anuncios, 38 columnas, 32 predictoras candidatas.
Entrada: 50,000; eliminados por superficies: 247.
Objetivo: price_rub en RUB, sin transformación.
Aplicado: tipos, mapas binarios, reglas físicas, tendencia, estacionalidad, LAG y partición mensual.
Pendiente por fold: imputación, One-Hot y eventual escalado / tratamiento estadístico.


,observaciones,fecha_min,fecha_max,porcentaje
dataset_split,,,,
train,34853,2020-01-01,2024-05-31,70.052057
validation,5258,2024-06-01,2025-01-31,10.568207
test,9642,2025-02-01,2026-04-29,19.379736


,nulos_pendientes
building_year,1266
n_listings_secondary_lag1,672
n_listings_newbuild_lag1,672
n_listings_rental_lag1,672
cbr_key_rate_pct_lag1,672
